In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [4]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [6]:
def create_outline(state:BlogState) -> BlogState:
    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate an detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # updated state
    state['outline'] = outline

    return state


In [7]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state


In [8]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog', create_blog)

# edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [9]:
initial_state = {'title':'Rise of Ai in India'}
final_state = workflow.invoke(initial_state)
print(final_state['outline'])

## Blog Outline: The Rise of AI in India - A Nation Embracing the Future

**I. Introduction: The AI Awakening in India**

    *   **A. Hook:** Start with a compelling statistic or anecdote showcasing AI's growing presence in India (e.g., a popular AI-powered app, a government initiative, a startling growth figure).
    *   **B. Defining AI (briefly):** A concise, layman's explanation of what Artificial Intelligence is and its core capabilities.
    *   **C. The Indian Context:** Why is India particularly ripe for AI adoption? (e.g., large young population, digital infrastructure growth, government push, diverse needs).
    *   **D. Thesis Statement/Blog's Purpose:** Clearly state what the blog will explore – the multifaceted rise of AI in India, its drivers, applications, challenges, and future potential.
    *   **E. Roadmap of the Blog:** Briefly outline the key sections the reader can expect.

**II. Drivers of AI's Ascent in India**

    *   **A. Government Initiatives and Policy Su